# Notebook 3 — Unseen-subject calibration, bin audits, clustered bootstrap CIs

Addresses Reviewer 1's calibration criticism and Reviewer 3's requests: **top-label reliability bin counts**, full Brier/ECE/MCE/NLL, the low-accuracy ~0.4 confidence bin, subject-aggregated calibration and 95% **subject-cluster bootstrap confidence intervals**.

**INPUT:** Executed Notebook 1 results ZIP (`AD_FTD_CN_GroupKFold_Reviewer_Results.zip`) or its results directory. The ZIP must contain `OOF_segment_predictions_subject_disjoint.csv`, `OOF_subject_predictions.csv` and `subject_split_manifest.csv`. This notebook does **not** retrain; it examines only predictions from held-out subjects, all 88 test subjects exactly once. The model is never calibrated/fitted on test results here, so scores describe *assessment* rather than post-hoc fitted calibration. It intentionally cannot explain the manuscript's original Figure 6a without original numerical OOF predictions, which were not fully exported in the supplied code archive.

Run in Colab **or locally**; supports `files.upload()` if the results ZIP isn't found.

In [ ]:
%pip -q install pandas numpy scipy scikit-learn matplotlib
import json, zipfile, warnings, math, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from scipy.stats import spearmanr
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)
CLASS_NAMES=['HC','AD','FTD']; P_COLS=['p_HC','p_AD','p_FTD']
INPUT_DIR=Path('/content/EEG_AD_FTD_CN_GroupKFold/results')
INPUT_ZIP=Path('/content/EEG_AD_FTD_CN_GroupKFold/AD_FTD_CN_GroupKFold_Reviewer_Results.zip')
OUTPUT=Path('/content/Notebook3_Subjectwise_Calibration_Results')
N_BOOT=1500; SEED=202631; N_BINS=10; EPS=1e-12
OUTPUT.mkdir(parents=True,exist_ok=True)

def safe_extract(zippath,destination):
    dest=Path(destination).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zippath) as z:
        for f in z.infolist():
            if not (dest/f.filename).resolve().is_relative_to(dest):
                raise ValueError('Unsafe path in ZIP: '+f.filename)
            if (f.external_attr >> 16)&0o170000 == 0o120000:
                raise ValueError('ZIP contains symlink')
        z.extractall(dest)

required=['OOF_segment_predictions_subject_disjoint.csv','OOF_subject_predictions.csv','subject_split_manifest.csv']
if not all((INPUT_DIR/p).exists() for p in required):
    if not INPUT_ZIP.exists():
        from google.colab import files
        print('Upload the RESULTS ZIP created by Notebook 1 (not the raw dataset):')
        result=files.upload()
        possibilities=[Path('/content')/p for p in result if p.lower().endswith('.zip')]
        if not possibilities:raise FileNotFoundError('No results ZIP provided')
        INPUT_ZIP=possibilities[0]
        del result
    INPUT_DIR=Path('/content/Notebook3_input')
    safe_extract(INPUT_ZIP,INPUT_DIR)
    if not all((INPUT_DIR/p).exists() for p in required):
        candidates=[p.parent for p in INPUT_DIR.rglob(required[0])]
        INPUT_DIR=next((p for p in candidates if all((p/q).exists() for q in required)),None)
        if INPUT_DIR is None:raise FileNotFoundError('The ZIP is missing required OOF outputs from Notebook 1')
print('Input:',INPUT_DIR,'Output:',OUTPUT)

In [ ]:
# Load and audit held-out prediction provenance. Abort on unknown subject leakage.
seg=pd.read_csv(INPUT_DIR/required[0]);sub=pd.read_csv(INPUT_DIR/required[1]);splits=pd.read_csv(INPUT_DIR/required[2])
for x in [seg,sub]:
    assert {'participant_id','fold','label',*P_COLS}.issubset(x.columns)
    assert x[P_COLS].notna().all().all()
    assert np.isfinite(x[P_COLS].to_numpy()).all()
    assert x[P_COLS].to_numpy().min()>=0 and x[P_COLS].to_numpy().max()<=1
    assert np.allclose(x[P_COLS].sum(axis=1),1,rtol=0,atol=1e-4)
    assert x['label'].isin([0,1,2]).all()
    assert x.groupby('participant_id')['fold'].nunique().max()==1
assert sub.participant_id.nunique()==88 and len(sub)==88
assert (sub.groupby('participant_id').size()==1).all()
assert sub.groupby('label').size().to_dict()=={0:29,1:36,2:23}
assert (seg.groupby('participant_id')['label'].nunique()==1).all()
assert set(seg.participant_id)==set(sub.participant_id)
assert splits.query("partition == 'test'").groupby('participant_id').size().eq(1).all()
assert set(splits.query("partition == 'test'").participant_id)==set(sub.participant_id)
merged=seg[['participant_id','fold','label']].drop_duplicates().merge(
    sub[['participant_id','fold','label']],on='participant_id',suffixes=('_segment','_subject'))
assert (merged.fold_segment==merged.fold_subject).all()
assert (merged.label_segment==merged.label_subject).all()
print('PASS — 88 unseen test subjects, class counts 29/36/23; one test fold each.')

In [ ]:
# Calibration function definitions. Multiclass Brier = mean(sum_k(p_k - onehot_k)^2).
def validate_arrays(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    assert p.ndim==2 and p.shape==(len(y),3)
    assert (y>=0).all() and (y<=2).all()
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-4)
    return y,p

def make_bins(y,p,n_bins=10,method='uniform'):
    y,p=validate_arrays(y,p)
    confidence=p.max(axis=1); correct=(p.argmax(axis=1)==y).astype(float)
    if method=='uniform':
        edges=np.linspace(0,1,n_bins+1)
    elif method=='quantile':
        edges=np.quantile(confidence,np.linspace(0,1,n_bins+1))
        edges[0]=min(0.,edges[0]);edges[-1]=1.
        edges=np.unique(edges)
    else:raise ValueError(method)
    idx=np.clip(np.searchsorted(edges,confidence,side='right')-1,0,len(edges)-2)
    table=[]
    for j in range(len(edges)-1):
        hit=idx==j;n=int(hit.sum())
        cm=float(confidence[hit].mean()) if n else np.nan
        ac=float(correct[hit].mean()) if n else np.nan
        gap=abs(ac-cm) if n else np.nan
        table.append({'bin':j,'lower_inclusive':edges[j],'upper_exclusive_except_final':edges[j+1],
            'n':n,'fraction':n/len(y),'mean_confidence':cm,'observed_accuracy':ac,
            'abs_gap':gap,'ece_contribution':gap*n/len(y) if n else 0})
    return pd.DataFrame(table)

def top_cal(y,p,n_bins=10,method='uniform'):
    y,p=validate_arrays(y,p)
    b=make_bins(y,p,n_bins,method)
    ece=float(b.ece_contribution.sum())
    mce=float(b.loc[b.n>0,'abs_gap'].max()) if (b.n>0).any() else np.nan
    return ece,mce,b

def scores(y,p):
    y,p=validate_arrays(y,p)
    truth=np.eye(3)[y];pred=p.argmax(axis=1)
    ece,mce,_=top_cal(y,p,N_BINS,'uniform')
    qece,qmce,_=top_cal(y,p,N_BINS,'quantile')
    brier=float(np.mean(np.sum((p-truth)**2,axis=1)))
    out={'n_samples':len(y),'accuracy':float(accuracy_score(y,pred)),
         'brier_multiclass':brier,'brier_half':brier/2,
         'ECE_uniform':ece,'MCE_uniform':mce,'ECE_quantile':qece,'MCE_quantile':qmce,
         'NLL':float(-np.log(np.clip(p[np.arange(len(y)),y],EPS,1.)).mean())}
    for k,name in enumerate(CLASS_NAMES):
        o=(y==k).astype(float)
        out['Brier_ovr_'+name]=float(np.mean((p[:,k]-o)**2))
        # OvR class-wise ECE measures predicted class probability over [0,1]
        ids=np.clip(np.floor(p[:,k]*N_BINS).astype(int),0,N_BINS-1)
        out['ECE_ovr_'+name]=float(sum((ids==j).mean()*abs(o[ids==j].mean()-p[ids==j,k].mean())
                for j in range(N_BINS) if (ids==j).any()))
    return out

score_rows=[]
for level,data in [('subject',sub),('segment_subject_held_out',seg)]:
    y=data.label.to_numpy(int);p=data[P_COLS].to_numpy(float)
    score_rows.append({'level':level,**scores(y,p)})
    for method in ['uniform','quantile']:
        for n_bins in [10,15]:
            tb=make_bins(y,p,n_bins,method)
            tb.to_csv(OUTPUT/f'{level}_{method}_{n_bins}bins_with_counts.csv',index=False)
            print(level,method,n_bins,'ECE',round(tb.ece_contribution.sum(),5),
                  'MCE',round(tb.loc[tb.n>0,'abs_gap'].max(),5),'sparsest populated bin',tb.loc[tb.n>0,'n'].min())
score_table=pd.DataFrame(score_rows)
score_table.to_csv(OUTPUT/'subject_vs_heldout_segment_calibration_scores.csv',index=False)
display(score_table)

In [ ]:
# Figure 6a anomaly audit: confidence ~0.4 bin; show the sample count explicitly.
for level,data in [('subject',sub),('segment_subject_held_out',seg)]:
    tbl=make_bins(data.label,data[P_COLS].to_numpy(),10,'uniform')
    focus=tbl[(tbl.lower_inclusive<.5)&(tbl.upper_exclusive_except_final>.3)]
    print('\n',level,'confidence bins spanning 0.3–0.5')
    display(focus[['lower_inclusive','upper_exclusive_except_final','n','mean_confidence',
                   'observed_accuracy','abs_gap']])
    sparse=tbl[(tbl.n>0)&(tbl.n<10)]
    print('Populated bins with <10 predictions:',len(sparse))
    if len(sparse):display(sparse)
print('Caution: these are NEW subject-held-out predictions; they cannot reproduce the original Figure 6a without its original numeric OOF table.')

## Cluster-robust confidence intervals (by unseen participant)

The segment-level bootstrap resamples *entire participants* rather than pretending thousands of correlated EEG windows are independent. The subject-level bootstrap samples one aggregated record per individual; both are **stratified by diagnostic class**. These intervals characterize cohort sampling variability and do not include full model-training uncertainty or independent-site generalization.

In [ ]:
# Bootstrap Brier, ECE and NLL on pooled subject predictions and on held-out segments.
all_subjects=np.array(sorted(sub.participant_id.unique()))
label_by_id=sub.set_index('participant_id').label.to_dict()
by_class=[np.array([sid for sid in all_subjects if label_by_id[sid]==k]) for k in [0,1,2]]
sub_by_id={s:g for s,g in sub.groupby('participant_id',sort=False)}
seg_by_id={s:g for s,g in seg.groupby('participant_id',sort=False)}
rng=np.random.default_rng(SEED)
metrics=['brier_multiclass','ECE_uniform','MCE_uniform','ECE_quantile','NLL','accuracy']
reps=[]
for b in range(N_BOOT):
    ids=np.concatenate([rng.choice(v,size=len(v),replace=True) for v in by_class])
    for level,lookup in [('subject',sub_by_id),('segment_subject_held_out',seg_by_id)]:
        boot=pd.concat([lookup[sid] for sid in ids],ignore_index=True)
        v=scores(boot.label.to_numpy(),boot[P_COLS].to_numpy())
        reps.append({'bootstrap':b,'level':level,**{k:v[k] for k in metrics}})
    if (b+1)%300==0:print('Completed',b+1,'subject-cluster bootstrap repeats')
bootstrap=pd.DataFrame(reps)
bootstrap.to_csv(OUTPUT/'subject_cluster_bootstrap_replicates.csv',index=False)
ci=[]
for _,row in score_table.iterrows():
    level=row.level;sample=bootstrap[bootstrap.level==level]
    for key in metrics:
        lo,hi=np.quantile(sample[key],[.025,.975])
        ci.append({'level':level,'metric':key,'point_estimate':row[key],
                   'CI95_low':float(lo),'CI95_high':float(hi),'resampling_unit':'subject',
                   'bootstrap_reps':N_BOOT})
ci_table=pd.DataFrame(ci)
ci_table.to_csv(OUTPUT/'bootstrap_95CI_Brier_ECE_MCE_NLL.csv',index=False)
display(ci_table)

In [ ]:
# Per-fold subject calibration and diagrams for class-wise + top-label reliability
fold_rows=[]
for fold,data in sub.groupby('fold'):
    fold_rows.append({'fold':int(fold),'level':'subject',**scores(data.label.values,data[P_COLS].values)})
for fold,data in seg.groupby('fold'):
    fold_rows.append({'fold':int(fold),'level':'segment_subject_held_out',**scores(data.label.values,data[P_COLS].values)})
pd.DataFrame(fold_rows).to_csv(OUTPUT/'foldwise_subject_heldout_calibration.csv',index=False)
for level,data in [('subject',sub),('segment_subject_held_out',seg)]:
    y=data.label.to_numpy(int);p=data[P_COLS].to_numpy(float)
    fig,ax=plt.subplots(figsize=(6.5,5.5))
    ax.plot([0,1],[0,1],'--',label='ideal')
    tb=make_bins(y,p,10,'uniform')
    m=tb.n>0
    ax.scatter(tb.loc[m,'mean_confidence'],tb.loc[m,'observed_accuracy'],
               s=np.maximum(18,2.2*tb.loc[m,'n'].to_numpy()**.5),label='observed top-label')
    ax.plot(tb.loc[m,'mean_confidence'],tb.loc[m,'observed_accuracy'],alpha=.7)
    for _,r in tb.loc[m].iterrows():
        ax.annotate(f'n={int(r.n)}',(r.mean_confidence,r.observed_accuracy),xytext=(4,4),
                    textcoords='offset points',fontsize=7)
    ax.set(xlim=(-.01,1.01),ylim=(-.05,1.05),xlabel='Mean softmax confidence',
           ylabel='Observed accuracy',title=f'Top-label calibration — {level}')
    ax.legend();fig.tight_layout()
    fig.savefig(OUTPUT/f'{level}_top_label_reliability.png',dpi=200);plt.show()
    fig,ax=plt.subplots(figsize=(6.5,5.3))
    ax.plot([0,1],[0,1],'--')
    for k,name in enumerate(CLASS_NAMES):
        pr=p[:,k];iscls=(y==k).astype(float)
        bins=np.clip((pr*10).astype(int),0,9)
        xc=[];yc=[]
        for j in range(10):
            use=bins==j
            if use.any():xc.append(pr[use].mean());yc.append(iscls[use].mean())
        ax.plot(xc,yc,'o-',label=f'{name} OvR')
    ax.set(xlim=(-.01,1.01),ylim=(-.05,1.05),xlabel='Predicted class probability',
           ylabel='Observed class frequency',title=f'Class-wise calibration — {level}')
    ax.legend();fig.tight_layout();fig.savefig(OUTPUT/f'{level}_classwise_reliability.png',dpi=200);plt.show()

In [ ]:
# Reproducibility manifest, ZIP with numerical outputs and figures.
manifest={'analysis':'Notebook 3: leakage-free held-out-subject calibration assessment',
          'n_subjects':int(sub.participant_id.nunique()),'n_segments':int(len(seg)),
          'N_BOOT':N_BOOT,'seed':SEED,'resampling_unit':'subject (class stratified)',
          'n_bins_default':N_BINS,'uncertainty_scope':'sampling of subjects from this single cohort',
          'no_temperature_fitted_on_test':True,
          'original_manuscript_Figure_6a_reproduced':False,
          'requires_Notebook1_executed_predictions':True}
(OUTPUT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=Path('/content/Notebook3_Subjectwise_Calibration_Results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.rglob('*'):
        if p.is_file():z.write(p,arcname=p.name)
print('Saved',archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass

### Interpretation for reviewer response

Report **subject-aggregated** calibration as primary, **subject-held-out segment calibration** as supplementary. Include raw bin counts, Brier/ECE/MCE/NLL and **clustered confidence intervals**. A large MCE in a sparsely occupied bin requires honest discussion even if ECE is small. **Never** fit Platt scaling, isotonic regression or temperature on the same OOF test labels unless a separate inner held-out calibration split is provided.